# Módulo 3 · Práctica: SVD, PCA y tensores

En este notebook comprimes una imagen con la **SVD**, visualizas **embeddings** con **PCA** y manejas tensores con las formas de un LLM real (batch × secuencia × d_model). Al final ves por qué **LoRA** (módulo 20) funciona: los cambios de bajo rango son baratos.

**Cómo trabajar:** ejecuta las celdas en orden. Las celdas `# TU CÓDIGO` son ejercicios; la celda `# Comprobación` siguiente te dice si están bien.

| Parte | Qué haces |
| --- | --- |
| 0 | Bases, rango e inversa en NumPy |
| 1 | Autovalores y autovectores |
| 2 | SVD y aproximación de bajo rango |
| 3 | Comprimir una imagen con SVD |
| 4 | Visualizar embeddings con PCA |
| 5 | Tensores y broadcasting con formas de LLM |
| 6 | Avance de LoRA |
| 7 | Retos |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.set_printoptions(precision=4, suppress=True)

## Parte 0 · Bases, rango e inversa

El **rango** es el número de columnas linealmente independientes. Una matriz cuadrada es invertible si y solo si su rango es máximo (determinante ≠ 0).

In [ ]:
M = np.array([[1., 2., 3.],
              [2., 4., 6.],
              [1., 0., 1.]])
print("rango(M) =", np.linalg.matrix_rank(M))   # la fila 2 es 2·fila 1
A = np.array([[2., 1.],
              [1., 1.]])
print("A^-1 =\n", np.linalg.inv(A))
print("A A^-1 =\n", A @ np.linalg.inv(A))

**Ejercicio 0.** El vector b = (3, 1) se escribe en la base {u₁ = (1, 1), u₂ = (1, −1)} como b = c₁u₁ + c₂u₂. Calcula las coordenadas `c` resolviendo un sistema (usa `np.linalg.solve`, no `inv`).

In [ ]:
U_base = np.array([[1., 1.],
                   [1., -1.]]).T   # columnas = vectores de la base
b = np.array([3., 1.])
# TU CÓDIGO
c = None

In [ ]:
# Comprobación
assert c is not None, "Calcula c"
assert np.allclose(U_base @ c, b), "U_base @ c debe reconstruir b"
print("Correcto. Coordenadas:", c)

## Parte 1 · Autovalores y autovectores

Un autovector v cumple Av = λv: la matriz solo lo estira. Para matrices **simétricas**, `np.linalg.eigh` devuelve autovalores reales y autovectores **ortonormales** (lo demuestras en la Profundización del documento).

In [ ]:
S = np.array([[2., 1.],
              [1., 2.]])
lam, Q = np.linalg.eigh(S)        # ordenados de menor a mayor
print("autovalores:", lam)
print("Q^T Q =\n", Q.T @ Q)        # identidad: ortonormales
print("Q diag(lam) Q^T =\n", Q @ np.diag(lam) @ Q.T)   # reconstruye S

**Ejercicio 1 (método de la potencia).** Multiplicar repetidamente un vector por S y normalizarlo converge al autovector del autovalor de mayor módulo. Implementa `potencia(S, n_iter)` que devuelva `(lam, v)`, con lam = vᵀSv (cociente de Rayleigh).

In [ ]:
def potencia(S, n_iter=100, seed=0):
    v = np.random.default_rng(seed).normal(size=S.shape[0])
    # TU CÓDIGO: repite v = S v / ||S v|| y calcula lam = v^T S v
    lam = None
    return lam, v

In [ ]:
# Comprobación
lam1, v1 = potencia(S)
assert lam1 is not None and abs(lam1 - 3) < 1e-8, "El autovalor dominante de S es 3"
assert np.allclose(np.abs(v1), np.ones(2) / np.sqrt(2)), "El autovector es ±(1, 1)/√2"
G = np.random.default_rng(1).normal(size=(6, 6)); G = G @ G.T
assert abs(potencia(G, 500)[0] - np.linalg.eigvalsh(G).max()) < 1e-6, "Falla con una matriz 6×6"
print("Correcto")

## Parte 2 · SVD y aproximación de bajo rango

Toda matriz A (m×n) se factoriza A = U Σ Vᵀ. Truncar a k valores singulares da la mejor aproximación de rango k (teorema de Eckart–Young):

$$A_k = \sum_{i=1}^{k} \sigma_i\, u_i v_i^\top, \qquad \|A - A_k\|_F = \sqrt{\sum_{i>k} \sigma_i^2}$$

In [ ]:
A = np.array([[3., 2., 2.],
              [2., 3., -2.]])
U, s, Vt = np.linalg.svd(A, full_matrices=False)
print("valores singulares:", s)          # [5, 3], como en el documento
print("reconstrucción correcta:", np.allclose(U @ np.diag(s) @ Vt, A))

**Ejercicio 2.** Implementa `rango_k(A, k)` que devuelva A_k usando la SVD.

In [ ]:
def rango_k(A, k):
    # TU CÓDIGO
    return None

In [ ]:
# Comprobación
A1 = rango_k(A, 1)
assert A1 is not None and np.allclose(A1, [[2.5, 2.5, 0], [2.5, 2.5, 0]]), "A_1 debe ser [[2,5 2,5 0],[2,5 2,5 0]]"
assert np.linalg.matrix_rank(A1) == 1
R = np.random.default_rng(0).normal(size=(30, 20))
sR = np.linalg.svd(R, compute_uv=False)
for k in [1, 5, 10]:
    err = np.linalg.norm(R - rango_k(R, k))
    assert np.isclose(err, np.sqrt((sR[k:] ** 2).sum())), f"Error de Frobenius incorrecto para k={k}"
print("Correcto: el error coincide con √(Σ σᵢ² para i > k)")

## Parte 3 · Comprimir una imagen con SVD

Generamos una imagen en escala de grises de 128×128 con NumPy: una letra «A», un anillo, un degradado y algo de ruido. No hace falta descargar nada.

In [ ]:
n = 128
yy, xx = np.mgrid[0:n, 0:n] / n
img = 0.3 * xx                                         # degradado horizontal
r = np.sqrt((xx - 0.72) ** 2 + (yy - 0.3) ** 2)
img += 0.6 * ((r > 0.13) & (r < 0.19))                 # anillo
def trazo(x0, y0, x1, y1, grosor=0.035):               # segmento grueso
    t = np.clip(((xx - x0) * (x1 - x0) + (yy - y0) * (y1 - y0)) / ((x1 - x0) ** 2 + (y1 - y0) ** 2), 0, 1)
    return np.hypot(xx - (x0 + t * (x1 - x0)), yy - (y0 + t * (y1 - y0))) < grosor
letra = trazo(0.12, 0.9, 0.3, 0.3) | trazo(0.3, 0.3, 0.48, 0.9) | trazo(0.19, 0.68, 0.41, 0.68)
img[letra] = 1.0
img += 0.03 * np.random.default_rng(0).normal(size=(n, n))
img = np.clip(img, 0, 1)
plt.imshow(img, cmap="gray"); plt.title("Imagen original 128×128"); plt.axis("off"); plt.show()

In [ ]:
Ui, si, Vti = np.linalg.svd(img, full_matrices=False)
plt.semilogy(si, ".-"); plt.xlabel("i"); plt.ylabel("σᵢ"); plt.title("Valores singulares de la imagen"); plt.show()

**Ejercicio 3.** Escribe:
- `ratio_compresion(m, n, k)`: números guardados con rango k, k(m + n + 1), divididos entre los de la imagen original, m·n.
- `k_para_energia(s, frac)`: el menor k tal que Σᵢ≤ₖ σᵢ² / Σ σᵢ² ≥ frac (pista: `np.cumsum` y `np.searchsorted` o un bucle).

In [ ]:
def ratio_compresion(m, n, k):
    # TU CÓDIGO
    return None

def k_para_energia(s, frac):
    # TU CÓDIGO
    return None

In [ ]:
# Comprobación
assert ratio_compresion(256, 256, 100) is not None and np.isclose(ratio_compresion(256, 256, 100), 51300 / 65536), "k(m+n+1)/(mn)"
assert k_para_energia(np.array([4., 3., 1.]), 0.95) == 2, "Con σ = (4, 3, 1), k = 2 retiene 25/26 ≈ 0,96"
assert k_para_energia(np.array([4., 3., 1.]), 0.97) == 3
k95 = k_para_energia(si, 0.95); k99 = k_para_energia(si, 0.99)
print(f"Correcto. k para 95 % de energía: {k95};  para 99 %: {k99}")

In [ ]:
fig, axs = plt.subplots(1, 4, figsize=(14, 4))
for ax, k in zip(axs, [1, 5, 20, 50]):
    Ak = rango_k(img, k)
    err = np.linalg.norm(img - Ak) / np.linalg.norm(img)
    ax.imshow(np.clip(Ak, 0, 1), cmap="gray"); ax.axis("off")
    ax.set_title(f"k={k} · {100*ratio_compresion(n, n, k):.0f} % datos · err {100*err:.1f} %")
plt.show()

**Pregunta.** ¿Por qué el degradado sale con k = 1 y las diagonales de la «A» necesitan muchos más términos? (Pista: en u vᵀ cada fila es múltiplo de la misma.)

## Parte 4 · Visualizar embeddings con PCA

Creamos embeddings sintéticos de dimensión 64 para 30 palabras de tres temas. Cada tema tiene un centro aleatorio y cada palabra añade ruido. Es un sustituto pequeño de unos embeddings preentrenados.

In [ ]:
temas = {
    "animales": ["gato", "perro", "loro", "vaca", "lobo", "oso", "pez", "rana", "toro", "foca"],
    "colores":  ["rojo", "azul", "verde", "gris", "rosa", "negro", "blanco", "ocre", "lila", "beige"],
    "números":  ["uno", "dos", "tres", "cuatro", "cinco", "seis", "siete", "ocho", "nueve", "diez"],
}
d = 64
rng = np.random.default_rng(42)
palabras, etiquetas, filas = [], [], []
for t, (tema, lista) in enumerate(temas.items()):
    centro = 2.0 * rng.normal(size=d)
    for p in lista:
        palabras.append(p); etiquetas.append(t)
        filas.append(centro + rng.normal(size=d))
E = np.array(filas)          # matriz de embeddings: una fila por palabra
etiquetas = np.array(etiquetas)
print("E.shape =", E.shape)

**Ejercicio 4.** Implementa `pca(X, k)` con la SVD:
1. Centra: Xc = X − media de las filas.
2. Xc = U Σ Vᵀ. Las componentes principales son las k primeras filas de Vᵀ.
3. Devuelve `Z = Xc @ Vt[:k].T` (proyecciones, forma n×k) y `var_exp` = σᵢ² / Σ σⱼ² para las k primeras.

In [ ]:
def pca(X, k):
    # TU CÓDIGO
    Z, var_exp = None, None
    return Z, var_exp

In [ ]:
# Comprobación
Z, var_exp = pca(E, 2)
assert Z is not None and Z.shape == (30, 2), "Z debe tener forma (30, 2)"
Xc = E - E.mean(axis=0)
C = Xc.T @ Xc / len(E)                         # covarianza
lamC = np.linalg.eigvalsh(C)[::-1]
assert np.allclose(var_exp, lamC[:2] / lamC.sum()), "var_exp debe coincidir con los autovalores de la covarianza"
assert np.allclose(np.var(Z, axis=0), lamC[:2]), "La varianza de cada proyección es un autovalor de C (relación SVD–PCA)"
Xm = np.array([[2, 2], [-2, -2], [1, -1], [-1, 1]], float)
assert np.allclose(np.abs(pca(Xm, 1)[0][:, 0]), [2 * np.sqrt(2), 2 * np.sqrt(2), 0, 0]), "Falla el ejemplo del documento"
print(f"Correcto. Varianza explicada por PC1 y PC2: {var_exp.round(3)}  (total {var_exp.sum():.3f})")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
for t, tema in enumerate(temas):
    m = etiquetas == t
    ax.scatter(Z[m, 0], Z[m, 1], label=tema)
for p, (x, y) in zip(palabras, Z):
    ax.annotate(p, (x, y), fontsize=8, alpha=0.7)
ax.set_xlabel("PC1"); ax.set_ylabel("PC2"); ax.legend(); ax.set_title("Embeddings 64-D proyectados con PCA")
plt.show()

**Pregunta.** Con 3 centros, ¿cuántas componentes bastan para separarlos? Mira `pca(E, 5)[1]`.

## Parte 5 · Tensores y broadcasting con formas de LLM

En un Transformer las activaciones tienen forma **(B, T, d)**: B secuencias, T tokens por secuencia, d = d_model. El operador `@` actúa sobre los **dos últimos ejes** y repite el resto (broadcasting).

In [ ]:
B_, T_, d_model, V_ = 2, 5, 8, 50
rng = np.random.default_rng(0)
emb = rng.normal(size=(V_, d_model))          # matriz de embeddings (V, d)
ids = rng.integers(0, V_, size=(B_, T_))      # tokens (B, T)
x = emb[ids]                                  # búsqueda de embeddings -> (B, T, d)
W = rng.normal(size=(d_model, d_model)); bias = rng.normal(size=d_model)
print("ids", ids.shape, "-> x", x.shape, "-> x @ W + bias", (x @ W + bias).shape)

**Ejercicio 5.** Con `x` de forma (B, T, d):
1. `xn`: resta a cada vector de token su media sobre d (el centrado de LayerNorm, módulo 11). Usa `keepdims=True`.
2. `Q = x @ Wq`, `K = x @ Wk` con matrices d×d, y `puntuaciones = Q Kᵀ / √d` con forma (B, T, T). Pista: `K.transpose(0, 2, 1)` o `np.swapaxes`.
3. Repite el paso 2 con `np.einsum` en `punt_einsum`.

In [ ]:
Wq = rng.normal(size=(d_model, d_model)); Wk = rng.normal(size=(d_model, d_model))
# TU CÓDIGO
xn = None
puntuaciones = None
punt_einsum = None

In [ ]:
# Comprobación
assert xn is not None and xn.shape == x.shape and np.allclose(xn.mean(axis=-1), 0), "Cada token debe tener media 0 en d"
assert puntuaciones is not None and puntuaciones.shape == (B_, T_, T_), "puntuaciones debe ser (B, T, T)"
ref = np.array([[(x[b] @ Wq)[i] @ (x[b] @ Wk)[j] for j in range(T_)] for b in range(B_) for i in range(T_)]).reshape(B_, T_, T_) / np.sqrt(d_model)
assert np.allclose(puntuaciones, ref), "Revisa el producto Q Kᵀ"
assert np.allclose(punt_einsum, ref), "Revisa la cadena de einsum"
print("Correcto. Estas son las puntuaciones de atención del módulo 13.")

## Parte 6 · Avance de LoRA

LoRA congela W (d×d) y aprende ΔW = B·A con B de d×r y A de r×d. La SVD te dice cuánto se pierde: si la actualización «real» tiene valores singulares que caen rápido, un r pequeño basta.

In [ ]:
d_big, r = 256, 8
rng = np.random.default_rng(5)
# Simulamos una actualización de pesos con espectro que decae
Uo, _ = np.linalg.qr(rng.normal(size=(d_big, d_big)))
Vo, _ = np.linalg.qr(rng.normal(size=(d_big, d_big)))
sig = 10 * 0.6 ** np.arange(d_big)
dW = Uo @ np.diag(sig) @ Vo.T
dW_r = rango_k(dW, r)
print(f"parámetros completos: {d_big**2}  · LoRA r={r}: {2*d_big*r}  ({100*2*r/d_big:.2f} %)")
print(f"error relativo con r={r}: {np.linalg.norm(dW - dW_r)/np.linalg.norm(dW):.4f}")

**Ejercicio 6.** Escribe `params_lora(d_in, d_out, r)` que devuelva el número de parámetros entrenables de LoRA, r·(d_in + d_out). Calcula `frac_4096` = fracción respecto a una matriz 4096×4096 con r = 8.

In [ ]:
# TU CÓDIGO
def params_lora(d_in, d_out, r):
    return None
frac_4096 = None

In [ ]:
# Comprobación
assert params_lora(4096, 4096, 8) == 65536, "8·(4096+4096) = 65.536"
assert np.isclose(frac_4096, 1 / 256), "65.536 / 16.777.216 = 1/256 ≈ 0,39 %"
print("Correcto")

## Parte 7 · Retos

1. **Imagen en color.** Crea una imagen RGB (n, n, 3) y comprime cada canal por separado. ¿Sale más a cuenta apilar los canales en una matriz (n, 3n)?
2. **PCA frente a ruido.** Sube el ruido de los embeddings de 1 a 3. ¿Cuándo dejan de separarse los clústeres en el plano PC1–PC2?
3. **Eckart–Young.** Genera 1.000 matrices aleatorias de rango 5 y comprueba que ninguna se acerca a R más que `rango_k(R, 5)`.

---
### Soluciones de referencia
Inténtalo antes de mirar.

In [ ]:
# Ejercicio 0
# c = np.linalg.solve(U_base, b)            # -> [2, 1]
# Ejercicio 1 (dentro de potencia)
#     for _ in range(n_iter):
#         v = S @ v
#         v = v / np.linalg.norm(v)
#     lam = v @ S @ v
# Ejercicio 2
#     U, s, Vt = np.linalg.svd(A, full_matrices=False)
#     return U[:, :k] @ np.diag(s[:k]) @ Vt[:k]
# Ejercicio 3
#     return k * (m + n + 1) / (m * n)
#     e = np.cumsum(s**2) / np.sum(s**2);  return int(np.searchsorted(e, frac - 1e-12) + 1)
# Ejercicio 4
#     Xc = X - X.mean(axis=0)
#     U, s, Vt = np.linalg.svd(Xc, full_matrices=False)
#     Z = Xc @ Vt[:k].T
#     var_exp = s[:k]**2 / np.sum(s**2)
# Ejercicio 5
# xn = x - x.mean(axis=-1, keepdims=True)
# Q, K = x @ Wq, x @ Wk
# puntuaciones = Q @ K.transpose(0, 2, 1) / np.sqrt(d_model)
# punt_einsum = np.einsum('bid,bjd->bij', Q, K) / np.sqrt(d_model)
# Ejercicio 6
#     return r * (d_in + d_out)
# frac_4096 = params_lora(4096, 4096, 8) / 4096**2